In [1]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Dataset

In [2]:
class KeypointDataset(Dataset):
    def __init__(self, csv_path):
        self.samples = []
        with open(csv_path, "r") as f:
            reader = csv.reader(f)
            for row in reader:
                label = int(row[0])
                points = list(map(float, row[1:]))
                self.samples.append((points, label))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        points, label = self.samples[idx]
        points = torch.tensor(points, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)
        return points, label


# Model

In [3]:
# Initialize the model
from keypoint_classifier import KeyPointModel

# Train

In [4]:
# Paths
dataset_path = "dataset/keypoint.csv"
dataset = KeypointDataset(dataset_path)
NUM_CLASSES = len(set([label for _, label in dataset.samples]))

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = KeyPointModel(NUM_CLASSES).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100

for epoch in range(EPOCHS):
	total_loss = 0
	
	for points, labels in train_loader:
		points, labels = points.to(device), labels.to(device)
		
		optimizer.zero_grad()
		
		logits = model(points)
		loss = nn.functional.cross_entropy(logits, labels)
		
		loss.backward()
		optimizer.step()
		
		total_loss += loss.item()
	
	cur_loss = total_loss / len(train_loader)
	print(f"Epoch {epoch+1}, loss: {cur_loss}")

model_save_path = "keypoint_model.pth"
torch.save(model.state_dict(), model_save_path)

Epoch 1, loss: 1.6198872089385987
Epoch 2, loss: 1.6141036748886108
Epoch 3, loss: 1.5974825620651245
Epoch 4, loss: 1.5812495231628418
Epoch 5, loss: 1.5733538627624513
Epoch 6, loss: 1.5661754369735719
Epoch 7, loss: 1.5359276056289672
Epoch 8, loss: 1.5193737506866456
Epoch 9, loss: 1.5086470603942872
Epoch 10, loss: 1.4884936332702636
Epoch 11, loss: 1.4580965042114258
Epoch 12, loss: 1.4041032552719117
Epoch 13, loss: 1.3932094812393188
Epoch 14, loss: 1.3531795024871827
Epoch 15, loss: 1.3138190031051635
Epoch 16, loss: 1.2837727546691895
Epoch 17, loss: 1.2456082344055175
Epoch 18, loss: 1.237163257598877
Epoch 19, loss: 1.2148757219314574
Epoch 20, loss: 1.1817859888076783
Epoch 21, loss: 1.1483348488807679
Epoch 22, loss: 1.1103612661361695
Epoch 23, loss: 1.0933905363082885
Epoch 24, loss: 1.0491589665412904
Epoch 25, loss: 1.0573194742202758
Epoch 26, loss: 1.038047695159912
Epoch 27, loss: 1.0298483610153197
Epoch 28, loss: 1.0017350554466247
Epoch 29, loss: 0.9829609274864